# Tutorial 14: Building Intuition for Static Layout Embeddings

SQuADDS can represent a physical layout as a numerical vector. That vector lets us
compare thousands of layouts quickly while still retaining interpretable information
about design parameters, geometric moments, and shape.

In this tutorial we will:

- inspect all **7,727 layouts** across `CapNInterdigitalTee`,
  `GeneralizedCapNInterdigital`, `CavityClawRouteMeander`, and `TransmonCross`;
- unpack the transparent `static-shape-v0` embedding model;
- use the new `StaticEmbeddingClient` and `SQuADDS_DB` bridge APIs;
- visualize the signed 96 x 96 shape tensor;
- explore the complete 7,727-layout embedding distribution interactively; and
- find similar and dissimilar components, then inspect their geometry side by side.

This follows Tutorial 1's learn-by-doing style. Every section starts with a concept,
exercises the API, and then visualizes what the numbers mean.

## 1. What is a static embedding?

An embedding is a fixed-length numerical description of an object. For v0, we choose a
deliberately simple and auditable model:

$$
\mathbf{e}_{v0} =
\operatorname{normalize}\left[
  \underbrace{\Sigma(\mathrm{design\ parameters})}_{1}
  \;\Vert\;
  \underbrace{\mathrm{geometric\ moments}}_{10}
  \;\Vert\;
  \underbrace{\mathrm{signed\ shape\ bitmap}}_{96 \times 96}
\right].
$$

The resulting vector has $1 + 10 + 9{,}216 = 9{,}227$ dimensions.

This is a **static baseline**, not a learned model. Its value is transparency: we can
point to every dimension and explain where it came from. A future learned v1 can be
compared against this baseline.

In [1]:
import json
import logging
import os

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
from huggingface_hub import hf_hub_download
from plotly.subplots import make_subplots

from squadds.core.db import SQuADDS_DB
from squadds.layouts import StaticEmbeddingClient, canonical_design_id

pio.renderers.default = "notebook_connected"
pd.set_option("display.max_colwidth", 80)
logging.getLogger("httpx").setLevel(logging.WARNING)

# The environment override lets this notebook be executed against a dataset PR.
# Normal users load the released data from main without changing anything.
EMBEDDING_REVISION = os.getenv("SQUADDS_EMBEDDING_REVISION", "main")
EMBEDDING_REPOSITORY = "SQuADDS/SQuADDS_Layout_Embeddings"
LAYOUT_REPOSITORY = "SQuADDS/SQuADDS_Layouts"
LAYOUT_REVISION = os.getenv("SQUADDS_LAYOUT_REVISION", "main")
DATABASE_REPOSITORY = "SQuADDS/SQuADDS_DB"

embedding_client = StaticEmbeddingClient(revision=EMBEDDING_REVISION)

## 2. Load embeddings through the public API

`StaticEmbeddingClient` downloads the versioned Parquet table lazily and caches it through
the Hugging Face Hub. `embeddings()` returns a copy as a Pandas DataFrame, while `get()`
retrieves one record by its stable `layout_id`.

In [2]:
embeddings = embedding_client.embeddings()

dataset_summary = (
    embeddings.groupby("component_name")
    .agg(layouts=("layout_id", "size"), unique_shapes=("shape_bitmap_sha256", "nunique"))
    .rename_axis("component family")
)
dataset_summary.loc["Combined"] = dataset_summary.sum()
dataset_summary

,layouts,unique_shapes
component family,,
CapNInterdigitalTee,894,894
CavityClawRouteMeander,1216,844
GeneralizedCapNInterdigital,3683,3683
TransmonCross,1934,1912
Combined,7727,7333


The combined row should report **7,727 simulation rows and 7,333 unique shape hashes**.
The rows contain 7,354 unique immutable GDS layouts; some simulation sweeps intentionally
repeat a geometry. The shape hash is useful for provenance and duplicate detection; it is
not used as a model feature.

Next, let us inspect the machine-readable schema rather than relying on a magic vector
length.

In [3]:
schema_path = hf_hub_download(
    repo_id=EMBEDDING_REPOSITORY,
    repo_type="dataset",
    filename="metadata/static-embedding-v0.schema.json",
    revision=EMBEDDING_REVISION,
)
with open(schema_path) as stream:
    schema = json.load(stream)

pd.DataFrame(
    [
        {
            "block": name,
            "offset": block["offset"],
            "dimensions": block["dimensions"],
            "meaning": {
                "parameter_sum": "unit-normalized, permutation-invariant design summary",
                "geometric_moments": "area, perimeter, aspect, occupancy, centroid, and moments",
                "shape_bitmap": "signed 96 x 96 functional-layout raster",
            }[name],
        }
        for name, block in schema["blocks"].items()
    ]
)

,block,offset,dimensions,meaning
0,parameter_sum,0,1,"unit-normalized, permutation-invariant design summary"
1,geometric_moments,1,10,"area, perimeter, aspect, occupancy, centroid, and moments"
2,shape_bitmap,11,9216,signed 96 x 96 functional-layout raster


In [4]:
block_frame = pd.DataFrame(
    {
        "block": ["parameter sum", "geometric moments", "shape bitmap"],
        "dimensions": [1, 10, 96 * 96],
    }
)
fig = px.bar(
    block_frame,
    x="block",
    y="dimensions",
    color="block",
    text="dimensions",
    log_y=True,
    title="The v0 vector is dominated by explicit shape pixels",
    color_discrete_sequence=["#D1495B", "#EDAe49", "#00798C"],
)
fig.update_traces(textposition="outside")
fig.update_layout(showlegend=False, yaxis_title="dimensions (log scale)")
fig.show()

### The three blocks

1. **Parameter sum (1 dimension).** Numeric values are parsed recursively, physical lengths
   are converted to micrometres, and the result is summed. Reordering dictionary keys or
   expressing `10um` as `0.01mm` does not change the result. The corpus-level value is
   standardized and passed through `tanh`.
2. **Geometric moments (10 dimensions).** These encode functional area and perimeter,
   bounding-box aspect ratio, occupancy, centroid, second central moments, and
   eccentricity. The block is standardized and normalized.
3. **Shape bitmap (9,216 dimensions).** Functional GDS polygons are cropped with preserved
   aspect ratio and rasterized at 96 x 96. Conductor pixels are positive, etch pixels are
   negative, generalized port layers have half weight, and background pixels are zero.

Each block is normalized before concatenation, and the complete vector is L2-normalized.
This prevents a physically larger layout from winning a comparison merely because it has
larger raw numbers.

## 3. Combine both NCap design datasets

The embedding table intentionally stores compact identifiers and model outputs. To make the
interactive plot pedagogical, we join it with design options from both source datasets.

The older component calls its finger width `cap_width`; the generalized component calls it
`finger_width`. We retain each original design and also create a few shared columns for
visualization.

In [5]:
DATASETS = {
    "CapNInterdigitalTee": "coupler-CapNInterdigitalTee-cap_matrix.json",
    "GeneralizedCapNInterdigital": "coupler-GeneralizedCapNInterdigital-cap_matrix.json",
}


def parse_number(value):
    """Parse the simple numeric and micrometre values used by these two datasets."""
    if isinstance(value, bool):
        return float(value)
    if isinstance(value, (int, float)):
        return float(value)
    if not isinstance(value, str):
        return np.nan
    cleaned = value.strip().replace("um", "")
    try:
        return float(cleaned)
    except ValueError:
        return np.nan


design_records = []
source_rows = {}
for component_name, filename in DATASETS.items():
    path = hf_hub_download(
        repo_id=DATABASE_REPOSITORY,
        repo_type="dataset",
        filename=filename,
    )
    with open(path) as stream:
        rows = json.load(stream)
    for row in rows:
        options = row["design"]["design_options"]
        design_id = canonical_design_id(component_name, options)
        source_rows[design_id] = row
        design_records.append(
            {
                "design_id": design_id,
                "finger_count": parse_number(options.get("finger_count")),
                "finger_length_um": parse_number(options.get("finger_length")),
                "finger_width_um": parse_number(
                    options.get("finger_width", options.get("cap_width"))
                ),
                "finger_gap_um": parse_number(
                    options.get("finger_gap_north_south", options.get("cap_gap"))
                ),
            }
        )

designs = pd.DataFrame(design_records)
# Cavity sweeps may repeat a simulation row for one canonical design/GDS identity.
geometry_path = hf_hub_download(
    repo_id=LAYOUT_REPOSITORY,
    repo_type="dataset",
    filename="metadata/geometry-features-v1.parquet",
    revision=LAYOUT_REVISION,
)
geometry = pd.read_parquet(geometry_path)

catalogue = (
    embeddings.merge(
        geometry.drop(columns=["artifact_id", "design_id", "component_name"]),
        on=["layout_id", "source_id"],
        validate="one_to_one",
    )
    .merge(designs, on="design_id", how="left", validate="many_to_one")
)
catalogue.groupby("component_name")[
    ["finger_count", "finger_length_um", "finger_width_um", "finger_gap_um"]
].agg(["min", "median", "max"])

finger_count              finger_length_um         \
                                     min median   max              min median   
component_name                                                                  
CapNInterdigitalTee                  1.0    5.0  10.0             15.9   30.9   
CavityClawRouteMeander               NaN    NaN   NaN              NaN    NaN   
GeneralizedCapNInterdigital          2.0    6.0  10.0              5.0   10.0   
TransmonCross                        NaN    NaN   NaN              NaN    NaN   

                                  finger_width_um              finger_gap_um  \
                              max             min median   max           min   
component_name                                                                 
CapNInterdigitalTee          60.9             4.9    8.9  14.9           1.1   
CavityClawRouteMeander        NaN             NaN    NaN   NaN           NaN   
GeneralizedCapNInterdigital  12.0             3.0    4.0   8.0           2.0   
TransmonCross                 NaN             NaN    NaN   NaN           NaN   

                                         
                            median  max  
component_name                           
CapNInterdigitalTee            3.1  6.1  
CavityClawRouteMeander         NaN  NaN  
GeneralizedCapNInterdigital    3.0  7.0  
TransmonCross                  NaN  NaN

## 4. See the high-dimensional distribution

Plotly cannot directly place 9,227 dimensions on a screen, so we create a deterministic,
fast two-dimensional sketch:

1. project all dimensions into 64 random directions;
2. run a small singular-value decomposition in that sketched space; and
3. collapse repeated simulation rows to their immutable GDS identity, then draw the
   resulting 7,354 physical layouts with Plotly's WebGL renderer.

This projection is only a visualization. Similarity search below always uses the complete
9,227-dimensional vectors.

Use the dropdown to color the same map by component family, functional area, bounding-box
shape, polygon count, or shape occupancy. These are defined for every component family.
Hover over a point to inspect its geometry; zoom, pan, and box-select to explore local
neighborhoods. The NCap-specific controls later in the tutorial retain the finger sweeps.

In [6]:
# The release has 7,727 simulation records but 7,354 immutable GDS geometries.
# Collapse provenance duplicates so dense sweeps do not visually dominate the map.
plot_catalogue = catalogue.drop_duplicates("layout_id").copy()
plot_embedding_matrix = np.vstack(plot_catalogue["embedding"]).astype(np.float32)

# A fixed seed makes this randomized PCA sketch reproducible while using every v0 dimension.
rng = np.random.default_rng(14)
projection = rng.normal(
    0.0, 1.0 / np.sqrt(64), size=(plot_embedding_matrix.shape[1], 64)
).astype(np.float32)
sketch = plot_embedding_matrix @ projection
sketch_mean = sketch.mean(axis=0, keepdims=True)
sketch -= sketch_mean
u, singular_values, sketch_basis = np.linalg.svd(sketch, full_matrices=False)
plot_catalogue["projection_x"] = u[:, 0] * singular_values[0]
plot_catalogue["projection_y"] = u[:, 1] * singular_values[1]

moments = np.vstack(plot_catalogue["geometric_moments"])
plot_catalogue["functional_log_area"] = moments[:, 0]
plot_catalogue["shape_occupancy"] = moments[:, 3]
plot_catalogue["log_total_area"] = np.log10(
    plot_catalogue["total_area_um2"].clip(lower=1e-6)
)
plot_catalogue["log_bbox_aspect"] = np.log10(
    plot_catalogue["bbox_aspect_ratio"].clip(lower=1e-6)
)

component_labels = {
    "CapNInterdigitalTee": "CapN",
    "GeneralizedCapNInterdigital": "Generalized NCap",
    "CavityClawRouteMeander": "Cavity claw",
    "TransmonCross": "Transmon cross",
}
component_codes = {name: index for index, name in enumerate(component_labels)}
plot_catalogue["component_label"] = plot_catalogue["component_name"].map(component_labels)
plot_catalogue["component_code"] = plot_catalogue["component_name"].map(component_codes)

color_options = {
    "Component family": (
        plot_catalogue["component_code"],
        [
            [0.0, "#D1495B"], [0.249, "#D1495B"],
            [0.25, "#00798C"], [0.499, "#00798C"],
            [0.5, "#E9C46A"], [0.749, "#E9C46A"],
            [0.75, "#6A4C93"], [1.0, "#6A4C93"],
        ],
        list(component_codes.values()),
        list(component_labels.values()),
    ),
    "Log total area (um2)": (plot_catalogue["log_total_area"], "Magma", None, None),
    "Functional log area": (
        plot_catalogue["functional_log_area"], "Cividis", None, None
    ),
    "Log bounding-box aspect ratio": (
        plot_catalogue["log_bbox_aspect"], "Turbo", None, None
    ),
    "Polygon count": (plot_catalogue["polygon_count"], "Viridis", None, None),
    "Shape occupancy": (plot_catalogue["shape_occupancy"], "Plasma", None, None),
}

initial_name = "Component family"
initial_color, initial_scale, initial_ticks, initial_labels = color_options[initial_name]
customdata = np.column_stack(
    [
        plot_catalogue["component_label"],
        plot_catalogue["total_area_um2"].round(1),
        plot_catalogue["bbox_width_um"].round(1),
        plot_catalogue["bbox_height_um"].round(1),
        plot_catalogue["polygon_count"],
        plot_catalogue["shape_occupancy"].round(3),
    ]
)
fig = go.Figure(
    go.Scattergl(
        x=plot_catalogue["projection_x"],
        y=plot_catalogue["projection_y"],
        mode="markers",
        marker={
            "color": initial_color,
            "colorscale": initial_scale,
            "size": 6,
            "opacity": 0.72,
            "colorbar": {
                "title": initial_name,
                "tickvals": initial_ticks,
                "ticktext": initial_labels,
            },
        },
        customdata=customdata,
        hovertemplate=(
            "<b>%{customdata[0]}</b><br>"
            "total area=%{customdata[1]} um2<br>"
            "bbox=%{customdata[2]} x %{customdata[3]} um<br>"
            "polygons=%{customdata[4]}<br>"
            "shape occupancy=%{customdata[5]}<extra></extra>"
        ),
    )
)

buttons = [
    {
        "label": label,
        "method": "update",
        "args": [
            {
                "marker.color": [color],
                "marker.colorscale": [colorscale],
                "marker.colorbar.title": [label],
                "marker.colorbar.tickvals": [tickvals],
                "marker.colorbar.ticktext": [ticktext],
            }
        ],
    }
    for label, (color, colorscale, tickvals, ticktext) in color_options.items()
]

fig.update_layout(
    title=f"Static v0 embedding map: {len(plot_catalogue):,} unique physical layouts",
    xaxis_title="randomized PCA sketch, axis 1",
    yaxis_title="randomized PCA sketch, axis 2",
    template="plotly_white",
    height=650,
    updatemenus=[
        {
            "buttons": buttons,
            "direction": "down",
            "x": 0.01,
            "y": 1.12,
            "xanchor": "left",
            "yanchor": "top",
        }
    ],
    annotations=[
        {
            "text": "Color points by:",
            "x": 0.01,
            "y": 1.18,
            "xref": "paper",
            "yref": "paper",
            "showarrow": False,
            "xanchor": "left",
        }
    ],
)
fig.show()

# Later similarity examples intentionally retain one row per simulation record.
catalogue = catalogue.merge(
    plot_catalogue.set_index("layout_id")[["projection_x", "projection_y"]],
    left_on="layout_id",
    right_index=True,
    how="left",
    validate="many_to_one",
)
embedding_matrix = np.vstack(catalogue["embedding"]).astype(np.float32)


### What should we notice?

- Component families occupy distinguishable regions because their functional topology and
  layer semantics differ.
- Coloring by finger count or length often produces smooth local trends. This is evidence
  that nearby geometric sweeps remain nearby in the embedding.
- The map is not a decision boundary or proof of physical equivalence. Projection compresses
  information, so trust full-dimensional similarity values and inspect the geometry.

## 5. Inspect the shape block directly

`shape_bitmap(layout_id)` recovers the normalized 96 x 96 shape block from the vector.
The common crop and signed layer semantics allow different physical sizes and component
implementations to be compared without losing topology.

In [7]:
representatives = (
    catalogue.loc[
        lambda frame: frame["component_name"].isin(
            ["CapNInterdigitalTee", "GeneralizedCapNInterdigital"]
        )
    ].assign(
        finger_distance=lambda frame: (
            frame["finger_count"]
            - frame.groupby("component_name")["finger_count"].transform("median")
        ).abs()
    )
    .sort_values(["component_name", "finger_distance", "finger_length_um"])
    .groupby("component_name", as_index=False)
    .head(1)
)

catalogue_by_layout = catalogue.drop_duplicates("layout_id").set_index("layout_id")

fig = make_subplots(
    rows=1,
    cols=2,
    subplot_titles=representatives["component_name"].tolist(),
    horizontal_spacing=0.08,
)
for column, (_, row) in enumerate(representatives.iterrows(), start=1):
    bitmap = embedding_client.shape_bitmap(row["layout_id"])
    fig.add_trace(
        go.Heatmap(
            z=bitmap,
            colorscale=[
                [0.0, "#D1495B"],
                [0.5, "#F7F7F2"],
                [1.0, "#00798C"],
            ],
            zmid=0,
            showscale=column == 2,
            colorbar={"title": "signed<br>shape"},
            hovertemplate="row=%{y}<br>column=%{x}<br>value=%{z:.4f}<extra></extra>",
        ),
        row=1,
        col=column,
    )
fig.update_yaxes(autorange="reversed", scaleanchor="x", scaleratio=1)
fig.update_layout(
    title="The API exposes the embedded shape tensor for visual inspection",
    height=480,
    template="plotly_white",
)
fig.show()

## 6. Similarity in the full embedding space

All complete v0 vectors have unit norm, so cosine similarity is simply a dot product:

$$
\operatorname{similarity}(\mathbf{e}_a,\mathbf{e}_b)
  = \frac{\mathbf{e}_a \cdot \mathbf{e}_b}
  {\|\mathbf{e}_a\|_2\|\mathbf{e}_b\|_2}
  = \mathbf{e}_a \cdot \mathbf{e}_b.
$$

Values closer to 1 indicate greater similarity under the v0 definition. This combines
parameter summary, global moments, and pixel-level shape. It does **not** claim that two
devices have identical capacitance or microwave response.

We will select a generalized component near the median finger count and ask the API for
its closest layouts across both families.

In [8]:
generalized = catalogue.loc[
    catalogue["component_name"] == "GeneralizedCapNInterdigital"
].copy()
anchor_row = generalized.iloc[
    (generalized["finger_count"] - generalized["finger_count"].median()).abs().argmin()
]
anchor_id = anchor_row["layout_id"]

nearest = pd.DataFrame(
    embedding_client.nearest(
        anchor_id,
        limit=8,
        component_name=None,  # Search both NCap datasets.
    )
).merge(
    catalogue.drop_duplicates("layout_id")[
        [
            "layout_id",
            "finger_count",
            "finger_length_um",
            "finger_width_um",
            "finger_gap_um",
        ]
    ],
    on="layout_id",
    how="left",
    validate="one_to_one",
)

nearest[
    [
        "component_name",
        "cosine_similarity",
        "finger_count",
        "finger_length_um",
        "finger_width_um",
        "finger_gap_um",
        "layout_id",
    ]
]

,component_name,cosine_similarity,finger_count,finger_length_um,finger_width_um,finger_gap_um,layout_id
0,GeneralizedCapNInterdigital,0.989883,6.0,12.0,6.0,5.0,layout:sha256:7dc2a3165b1ca5c40f3ea4afc1078a2303bd86ecabf1ac76a07c9cca6358ee1b
1,GeneralizedCapNInterdigital,0.981906,6.0,10.0,6.0,3.0,layout:sha256:3fb68f67659fbb8d6fc630c06033392aff8ae660eb43a3dfdd0ea5acfd32342d
2,GeneralizedCapNInterdigital,0.981641,6.0,9.0,5.0,4.0,layout:sha256:f2f677f6db9786c14b73524f870a7d80ddb26f03e57d7c82057d51643fdaa37c
3,GeneralizedCapNInterdigital,0.981610,6.0,12.0,6.0,5.0,layout:sha256:c791abd31adb2bd350ca2dc7608f06aba884403ba39e7dfee35351885e563a01
4,GeneralizedCapNInterdigital,0.978265,6.0,8.0,4.0,3.0,layout:sha256:9a8b64976f8d50b2225ff82ee5f08c0631aba40efbeebd4d78bed129b620029b
5,GeneralizedCapNInterdigital,0.978135,6.0,12.0,6.0,5.0,layout:sha256:f2989d57929138b516088742763570672c217cbc848e9f0b2c902e72c2d72e49
6,GeneralizedCapNInterdigital,0.977148,6.0,5.0,3.0,2.0,layout:sha256:09961a56fb5068720e385760dccf860b3f0003710ffd16e40235f6417f90482c
7,GeneralizedCapNInterdigital,0.974645,6.0,9.0,5.0,4.0,layout:sha256:af1172b0ba78ce37392001f05fa9fa1f68cd9ceb6599814739cb00a58a1b77db


The `embedding` column is intentionally omitted from nearest-neighbor responses by default:
returning thousands of 9,227-value vectors would waste bandwidth. Pass
`include_embeddings=True` only when a downstream calculation needs them.

Now let us compare the query with its closest result, its nearest layout from the other
component family, and the least-similar layout in the combined collection.

In [9]:
query_vector = np.asarray(embedding_client.get(anchor_id)["embedding"], dtype=np.float32)
similarities = embedding_matrix @ query_vector
catalogue["similarity_to_anchor"] = similarities
catalogue_by_layout = catalogue.drop_duplicates("layout_id").set_index("layout_id")

closest_id = nearest.iloc[0]["layout_id"]
other_family = catalogue.loc[
    catalogue["component_name"] != anchor_row["component_name"]
].nlargest(1, "similarity_to_anchor").iloc[0]
distant = catalogue.nsmallest(1, "similarity_to_anchor").iloc[0]

comparison_ids = [
    ("Query", anchor_id),
    ("Closest", closest_id),
    ("Closest other family", other_family["layout_id"]),
    ("Most dissimilar", distant["layout_id"]),
]

fig = make_subplots(
    rows=1,
    cols=4,
    subplot_titles=[
        f"{label}<br>similarity={catalogue_by_layout.loc[layout_id, 'similarity_to_anchor']:.3f}"
        for label, layout_id in comparison_ids
    ],
    horizontal_spacing=0.03,
)
for column, (_, layout_id) in enumerate(comparison_ids, start=1):
    fig.add_trace(
        go.Heatmap(
            z=embedding_client.shape_bitmap(layout_id),
            colorscale=[
                [0.0, "#D1495B"],
                [0.5, "#F7F7F2"],
                [1.0, "#00798C"],
            ],
            zmid=0,
            showscale=column == 4,
            colorbar={"title": "signed<br>shape"},
            hovertemplate="row=%{y}<br>column=%{x}<br>value=%{z:.4f}<extra></extra>",
        ),
        row=1,
        col=column,
    )
fig.update_yaxes(autorange="reversed", scaleanchor="x", scaleratio=1)
fig.update_layout(
    title="Similarity becomes intuitive when we inspect the encoded shapes",
    height=410,
    template="plotly_white",
)
fig.show()

The closest shape should preserve the anchor's major topology and proportions. The nearest
member of the other family is informative but usually less similar, while the most
dissimilar example visibly changes the geometry.

We can also color the full distribution by similarity to our query. This turns nearest
neighbors into a continuous landscape rather than a top-k list.

In [10]:
fig = px.scatter(
    catalogue,
    x="projection_x",
    y="projection_y",
    color="similarity_to_anchor",
    hover_name="component_name",
    hover_data={
        "finger_count": True,
        "finger_length_um": ":.2f",
        "finger_width_um": ":.2f",
        "projection_x": False,
        "projection_y": False,
    },
    render_mode="webgl",
    color_continuous_scale="Turbo",
    title="Similarity to one query across both NCap datasets",
    labels={"similarity_to_anchor": "cosine similarity"},
)
fig.add_trace(
    go.Scatter(
        x=[anchor_row["projection_x"]],
        y=[anchor_row["projection_y"]],
        mode="markers",
        marker={"symbol": "star", "size": 18, "color": "black"},
        name="query",
        hovertemplate="query<extra></extra>",
    )
)
fig.update_layout(height=620, template="plotly_white")
fig.show()

## 7. Embedding algebra: can a geometric change be transferred?

Vector representations invite algebra. If the change from layout $A$ to layout $B$
represents "add one finger," we can ask whether that same change transfers to a different
starting geometry $C$:

$$
\Delta_{\mathrm{finger}} = E(B) - E(A), \qquad
E_{\mathrm{prediction}} = E(C) + \Delta_{\mathrm{finger}}.
$$

We then normalize the prediction and decode it by finding the closest real layout.

This is an **experiment**, not a promise built into v0. The model was not trained with an
analogy objective, and its single parameter-sum dimension loses information. A successful
controlled example shows what embedding algebra means; it does not prove that every
geometric edit will transfer.

We use four real `CapNInterdigitalTee` designs from the combined catalogue:

- $A$: one finger, length 35.9 um, width 12.9 um;
- $B$: the same geometry with two fingers;
- $C$: one finger, length 40.9 um, width 13.9 um; and
- $D$: the expected transferred result, namely the geometry of $C$ with two fingers.

In [11]:
ALGEBRA_LAYOUTS = {
    "A: source (1 finger)": "layout:sha256:a3b2b8239d65c1f37b2c0254c53b94e2c873da45743c751469e5866c910fa0f6",
    "B: source + finger": "layout:sha256:3f2ef6499ffe6c474c8c6a603edae9f4b08837b0a2788612e39559682fc81ec6",
    "C: transfer base (1 finger)": "layout:sha256:989fa56a637585664ef2b77af8d171cc03e7e59231bc4c2112c406d54de7aa17",
    "D: expected (2 fingers)": "layout:sha256:e3105c3113de0326dadcc80b9c7a308785a0d21af16e5e38405438d0ab179ec2",
}

algebra_records = {
    label: embedding_client.get(layout_id)
    for label, layout_id in ALGEBRA_LAYOUTS.items()
}
algebra_vectors = {
    label: np.asarray(record["embedding"], dtype=np.float32)
    for label, record in algebra_records.items()
}

finger_delta = (
    algebra_vectors["B: source + finger"]
    - algebra_vectors["A: source (1 finger)"]
)
raw_prediction = algebra_vectors["C: transfer base (1 finger)"] + finger_delta
normalized_prediction = raw_prediction / np.linalg.norm(raw_prediction)

algebra_similarities = embedding_matrix @ normalized_prediction
decoded_index = int(np.argmax(algebra_similarities))
decoded = catalogue.iloc[decoded_index]
expected_id = ALGEBRA_LAYOUTS["D: expected (2 fingers)"]
expected_index = int(np.flatnonzero(catalogue["layout_id"].to_numpy() == expected_id)[0])
expected_rank = int(
    np.sum(algebra_similarities > algebra_similarities[expected_index]) + 1
)

algebra_table = []
for label, layout_id in ALGEBRA_LAYOUTS.items():
    design_id = catalogue.set_index("layout_id").loc[layout_id, "design_id"]
    options = source_rows[design_id]["design"]["design_options"]
    algebra_table.append(
        {
            "role": label,
            "finger_count": options["finger_count"],
            "finger_length_um": parse_number(options["finger_length"]),
            "finger_width_um": parse_number(options["cap_width"]),
            "cosine_to_prediction": float(
                algebra_similarities[
                    np.flatnonzero(catalogue["layout_id"].to_numpy() == layout_id)[0]
                ]
            ),
        }
    )

pd.DataFrame(algebra_table).assign(
    decoded_layout=decoded["layout_id"],
    expected_rank=expected_rank,
)

,role,finger_count,finger_length_um,finger_width_um,cosine_to_prediction,decoded_layout,expected_rank
0,A: source (1 finger),1,35.9,12.9,0.860591,layout:sha256:e3105c3113de0326dadcc80b9c7a308785a0d21af16e5e38405438d0ab179ec2,1
1,B: source + finger,2,35.9,12.9,0.979832,layout:sha256:e3105c3113de0326dadcc80b9c7a308785a0d21af16e5e38405438d0ab179ec2,1
2,C: transfer base (1 finger),1,40.9,13.9,0.890740,layout:sha256:e3105c3113de0326dadcc80b9c7a308785a0d21af16e5e38405438d0ab179ec2,1
3,D: expected (2 fingers),2,40.9,13.9,0.991086,layout:sha256:e3105c3113de0326dadcc80b9c7a308785a0d21af16e5e38405438d0ab179ec2,1


The expected design ranks first across both NCap datasets in this example. Before looking at
the bitmaps, we can visualize the arithmetic in the same linear projection used earlier.

The two arrows are equal before final normalization: $A \rightarrow B$ is copied to start
at $C$. The star marks the real layout recovered by nearest-neighbor decoding.

In [12]:
def project_embedding(vector):
    sketched = vector @ projection - sketch_mean.ravel()
    return sketched @ sketch_basis[:2].T


algebra_xy = {
    label: project_embedding(vector)
    for label, vector in algebra_vectors.items()
}
algebra_xy["P: C + (B - A)"] = project_embedding(raw_prediction)

fig = go.Figure()
fig.add_trace(
    go.Scattergl(
        x=catalogue["projection_x"],
        y=catalogue["projection_y"],
        mode="markers",
        marker={"size": 4, "color": "#C9D6D8", "opacity": 0.24},
        name="combined catalogue",
        hoverinfo="skip",
    )
)
for start, end, color, name in [
    ("A: source (1 finger)", "B: source + finger", "#D1495B", "learned edit: +1 finger"),
    ("C: transfer base (1 finger)", "P: C + (B - A)", "#00798C", "transferred edit"),
]:
    fig.add_trace(
        go.Scatter(
            x=[algebra_xy[start][0], algebra_xy[end][0]],
            y=[algebra_xy[start][1], algebra_xy[end][1]],
            mode="lines+markers+text",
            line={"color": color, "width": 4},
            marker={"size": 11, "color": color},
            text=[start.split(":")[0], end.split(":")[0]],
            textposition=(
                ["bottom left", "top left"]
                if start.startswith("A:")
                else ["bottom right", "top right"]
            ),
            name=name,
            hovertemplate="%{text}<extra></extra>",
        )
    )
    fig.add_annotation(
        x=algebra_xy[end][0],
        y=algebra_xy[end][1],
        ax=algebra_xy[start][0],
        ay=algebra_xy[start][1],
        xref="x",
        yref="y",
        axref="x",
        ayref="y",
        text="",
        showarrow=True,
        arrowhead=3,
        arrowsize=1.4,
        arrowwidth=3,
        arrowcolor=color,
    )

decoded_xy = catalogue.loc[decoded_index, ["projection_x", "projection_y"]]
fig.add_trace(
    go.Scatter(
        x=[decoded_xy["projection_x"]],
        y=[decoded_xy["projection_y"]],
        mode="markers+text",
        marker={"symbol": "star", "size": 20, "color": "#EDAe49", "line": {"width": 1}},
        text=["decoded D"],
        textposition="bottom center",
        name="nearest real layout",
        hovertemplate=(
            "decoded layout<br>cosine=%{customdata:.4f}<extra></extra>"
        ),
        customdata=[float(algebra_similarities[decoded_index])],
    )
)
local_points = np.vstack(
    [*algebra_xy.values(), decoded_xy.to_numpy(dtype=float)]
)
x_padding = max(float(np.ptp(local_points[:, 0])) * 0.25, 0.01)
y_padding = max(float(np.ptp(local_points[:, 1])) * 0.25, 0.01)

fig.update_layout(
    title="Vector arithmetic transfers the +1 finger direction",
    xaxis={
        "title": "randomized PCA sketch, axis 1",
        "range": [
            float(local_points[:, 0].min() - x_padding),
            float(local_points[:, 0].max() + x_padding),
        ],
    },
    yaxis={
        "title": "randomized PCA sketch, axis 2",
        "range": [
            float(local_points[:, 1].min() - y_padding),
            float(local_points[:, 1].max() + y_padding),
        ],
    },
    height=560,
    template="plotly_white",
)
fig.show()

In [13]:
shape_offset = schema["blocks"]["shape_bitmap"]["offset"]
predicted_shape = normalized_prediction[shape_offset:].reshape(96, 96)
algebra_shapes = [
    ("A: 1 finger", embedding_client.shape_bitmap(ALGEBRA_LAYOUTS["A: source (1 finger)"])),
    ("B: 2 fingers", embedding_client.shape_bitmap(ALGEBRA_LAYOUTS["B: source + finger"])),
    ("C: new 1-finger base", embedding_client.shape_bitmap(ALGEBRA_LAYOUTS["C: transfer base (1 finger)"])),
    ("P: algebra result", predicted_shape),
    ("D: nearest real layout", embedding_client.shape_bitmap(decoded["layout_id"])),
]

fig = make_subplots(
    rows=1,
    cols=5,
    subplot_titles=[title for title, _ in algebra_shapes],
    horizontal_spacing=0.025,
)
for column, (_, bitmap) in enumerate(algebra_shapes, start=1):
    fig.add_trace(
        go.Heatmap(
            z=bitmap,
            colorscale=[[0.0, "#D1495B"], [0.5, "#F7F7F2"], [1.0, "#00798C"]],
            zmid=0,
            showscale=column == 5,
            colorbar={"title": "signed<br>shape"},
            hovertemplate="row=%{y}<br>column=%{x}<br>value=%{z:.4f}<extra></extra>",
        ),
        row=1,
        col=column,
    )
fig.update_yaxes(autorange="reversed", scaleanchor="x", scaleratio=1)
fig.update_layout(
    title=(
        "E(C) + [E(B) - E(A)] decodes to D "
        f"(cosine={algebra_similarities[decoded_index]:.3f})"
    ),
    height=390,
    template="plotly_white",
)
fig.show()

### How to interpret this result

The algebraic shape `P` is not itself a manufacturable GDS file. Pixel subtraction leaves a
residual of the original context, which is visible in the signed tensor. Nearest-neighbor
decoding projects that arithmetic result back onto the catalogue of real designs and
recovers `D`.

This gives us two complementary tools:

- **Similarity** asks, "Which existing layouts resemble this layout?"
- **Algebra** asks, "Can a direction associated with a geometric edit be transferred?"

A rigorous future evaluation should form many held-out $(A,B,C,D)$ quadruples and report
top-k retrieval accuracy. A learned v1 could explicitly optimize this kind of
compositional behavior.

## 8. Connect a simulation row to its layout and embedding

Tutorial 1 introduced `SQuADDS_DB` rows. The new bridge methods let the same row lead to a
stable layout identity and then to its v0 vector. This works even for legacy
`coupler_type` labels.

In [14]:
source_row = source_rows[anchor_row["design_id"]]

layout_reference = SQuADDS_DB.get_layout_ref(source_row)
embedding_record = SQuADDS_DB.get_layout_embedding(
    source_row,
    embedding_client=embedding_client,
)

{
    "component": layout_reference.component_name,
    "design_id": layout_reference.design_id,
    "layout_id": layout_reference.layout_id,
    "embedding_model": embedding_record["embedding_model"],
    "embedding_dimensions": len(embedding_record["embedding"]),
}

{'component': 'GeneralizedCapNInterdigital',
 'design_id': 'design:sha256:2d8384b55dd567b3a02c724d4b873d164cc2213aaa77943b1da21dc75a2a9bbd',
 'layout_id': 'layout:sha256:22639c7d6d09e87ec522a39b30233aa5fa1d2ca59fadeef1a5ef270023ce2884',
 'embedding_model': 'static-shape-v0',
 'embedding_dimensions': 9227}

The same capabilities are available to AI agents through the SQuADDS MCP tools:

- `get_layout_embedding` fetches one complete v0 record;
- `find_similar_layouts` performs nearest-neighbor search;
- `get_layout_geometry` and `get_layout_layers` inspect the source GDS geometry; and
- `download_layout_gds` retrieves the checksum-verified artifact.

## 9. Responsible interpretation and next steps

`static-shape-v0` is intentionally a proof-of-concept:

- A parameter sum is permutation-invariant but cannot distinguish every parameter
  combination.
- A 96 x 96 bitmap is transparent but relatively large and can alias very fine geometry.
- Geometric similarity is not the same as electromagnetic or fabrication similarity.

A strong learned v1 could use multi-channel signed-distance fields or a polygon graph
encoder, then train contrastively with geometry and simulation targets. Until that model
demonstrates measurable improvement, v0 provides a reproducible, inspectable baseline.

**Try it yourself:** choose a different `anchor_id`, constrain `nearest()` with
`component_name`, or add another design option to the color dropdown. The most useful
intuition comes from repeatedly moving between the distribution, numerical similarity,
and actual shape.